In [1]:
!which python
!python --version

/home/agentolek/anaconda3/envs/ssne/bin/python
Python 3.13.12


In [4]:
import pandas as pd
import numpy as np

In [5]:
def rmsle(y_true,y_pred):
    n = len(y_true)
    msle = np.mean([(np.log(max(y_pred[i],0) + 1) - np.log(y_true[i] + 1)) ** 2.0 for i in range(n)])
    return np.sqrt(msle)

In [6]:
# check for NAs
data_df = pd.read_csv("data.csv")
print(f"Number of records: {len(data_df)}")
print(f"Number of rows with NAs: {data_df.isna().sum().sum()}")

Number of records: 10886
Number of rows with NAs: 0


In [7]:
# look at correlations to casual, registered, and cnt



In [8]:
from typing import Any

import torch
from torch import nn
# create simple neural network
class RegressionModel(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = 32, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)

        self.layers = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)

ModuleNotFoundError: No module named 'torch'

In [ ]:
# create Dataset class
class PandasBikeDataset(torch.utils.data.Dataset):
    def __init__(self, 
                 data_df: pd.DataFrame, 
                 pred_series: pd.Series, 
                 exclude_columns: list[str] | None = None) -> None:
        super().__init__()

        if exclude_columns is not None:
            self.data_df = data_df.drop(exclude_columns, axis=1)
        else:
            self.data_df = data_df
        
        self.labels = torch.Tensor(pred_series).to(dtype=torch.float)
    
    def __len__(self):
        return len(self.data_df)
    
    def __getitem__(self, index) -> Any:
        data = np.array(self.data_df.iloc[index].values)
        data = torch.from_numpy(data).to(dtype=torch.float)
        label = self.labels[index]
        return data, label

In [ ]:
data_df.columns

Index(['instant', 'dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday',
       'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
       'casual', 'registered', 'cnt'],
      dtype='str')

In [ ]:
from sklearn.model_selection import train_test_split
# create Dataset
exclude_columns = ["instant", "casual", "registered", "cnt", "dteday"]
# exclude_columns = list(data_df.columns)
# exclude_columns.remove("casual")
# exclude_columns.remove("registered")
dataset = PandasBikeDataset(data_df=data_df,
                            pred_series=data_df["cnt"],
                            exclude_columns=exclude_columns)
                            
train_dataset, test_dataset = train_test_split(dataset, test_size=0.5, shuffle=True)

len(train_dataset)

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
from tqdm import tqdm

model = RegressionModel(input_size=len(train_dataset[0][0]))
device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

# dataloaders
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

# hyperparameters
start_lr = 10e-1
epochs = 3

# optimizer, loss_fn
optim = torch.optim.AdamW(model.parameters(), lr=start_lr)
loss_fn = nn.MSELoss()

for epoch in tqdm(range(epochs)):
    running_loss = 0.0

    for i, data in enumerate(train_dataloader):

        X, y = data
        X, y = X.to(device), y.to(device).unsqueeze(dim=1)

        optim.zero_grad()

        preds = model(X)
        loss = loss_fn(preds, y)

        loss.backward()

        optim.step()
        running_loss += loss
        if (i+1) % 20 == 0:
            print('  batch {} loss: {}'.format(i + 1, running_loss / i))

100%|██████████| 3/3 [00:00<00:00, 16.02it/s]

  batch 20 loss: 67494.6953125
  batch 40 loss: 61230.8125
  batch 60 loss: 51075.78515625
  batch 80 loss: 44896.26171875
  batch 20 loss: 26688.111328125
  batch 40 loss: 25303.513671875
  batch 60 loss: 25807.66015625
  batch 80 loss: 26227.23828125
  batch 20 loss: 25373.625
  batch 40 loss: 24808.28125
  batch 60 loss: 25315.654296875
  batch 80 loss: 25699.97265625


In [ ]:
# generate predictions on evaluation data using model
test_dataloader = torch.utils.data.DataLoader(test_dataset)

preds = torch.Tensor([])
targets = torch.Tensor([])

with torch.inference_mode():
    for X, y in tqdm(test_dataloader):
        X, y = X.to(device), y.to(device)

        curr_preds = model(X)
        preds = torch.cat((preds, curr_preds), dim=0)
        targets = torch.cat((targets, y), dim=0)


100%|██████████| 5443/5443 [00:00<00:00, 19128.90it/s]


In [ ]:
pd.DataFrame(preds).to_csv("muchomorki.csv", index=False, header=None)

In [ ]:
rmsle(targets.numpy(), preds.numpy())

np.float32(1.3195908)